In [1]:
import requests
import pandas as pd
import numpy as np
import holidays
from datetime import datetime, timedelta
from sklearn.preprocessing import MinMaxScaler
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout
from tensorflow.keras.optimizers import Adam
import matplotlib.pyplot as plt

I0000 00:00:1788915512.844261    8090 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: SSE4.1 SSE4.2 AVX AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


In [2]:
data1 = pd.read_csv('../data/processed_data.csv')

In [67]:
data1['toll_10_minute_block'] = pd.to_datetime(data1['toll_10_minute_block'])

In [77]:
def fetch_traffic_data(DATA_PATH,full_pull=False):
    """
    Fetch CRZ traffic data from NY Gov API
    Data is pulled -3 days and on from current max date in data.
    """
    # get range start date
    current_data = pd.read_csv(
        DATA_PATH
    )

    current_data['toll_10_minute_block'] = pd.to_datetime(
        current_data['toll_10_minute_block']
    )

    current_max_date = (
        current_data['toll_10_minute_block'].max()
    )

    api_start_date = (
        current_max_date - pd.Timedelta(days=3)
    )

    print(
        f"Current data max date: {current_max_date}."
    )

    if full_pull == False:
            
        print(
            f"API will pull data beginning on: {api_start_date}"
        )

    else:

        print("Initializing full data pull.")

    # api setup

    url = "https://data.ny.gov/resource/t6yz-b64h.json"

    chunk_size = 50_000
    offset = 0

    chunks = []

    # format date for Socrata API

    api_date_string = api_start_date.strftime(
        "%Y-%m-%dT%H:%M:%S"
    )

    # fetch new data

    while True:

        print(

            f"Fetching rows "
            f"{offset + 1:,}-"
            f"{offset + chunk_size:,} ..."

        )

        if full_pull == False:

            params = {
                "$limit": chunk_size,
                "$offset": offset,

                "$where": (
                    f"toll_10_minute_block > '{api_date_string}'"
                ),

                "$order":"toll_10_minute_block ASC"
            }

        else:

            params = {
                "$limit": chunk_size,
                "$offset": offset,

                "$order":"toll_10_minute_block ASC"
            }

        response = requests.get(
            url,
            params=params,
            timeout=60
        )

        # raise error if unsuccesful response
        response.raise_for_status()
        
        api_data = response.json()

        if len(api_data) == 0:

            print("No more data to fetch.")

            break

        chunks.append(
            pd.DataFrame(api_data)
        )

        offset += chunk_size
    
    # combine chunks
    
    if not chunks:

        print("No new API data found.")
        return pd.DataFrame()

    df_all = pd.concat(
        chunks,
        ignore_index=True
    )

    # convert timestamp immediately
    df_all['toll_10_minute_block'] = pd.to_datetime(df_all['toll_10_minute_block'])

    print(
        f"Fetched {len(df_all):,} rows."
    )

    print(
        "API date range:",
        df_all["toll_10_minute_block"].min(),
        "to",
        df_all["toll_10_minute_block"].max()
    )

    return df_all

In [19]:
new_data.to_csv('../data/crz_data_pull_20260908.csv',index=None)

In [11]:
DATA_PATH = "../data/processed_data.csv"
new_data = fetch_traffic_data(DATA_PATH)

Current data max date: 2026-08-22 23:50:00.
API will pull data beginning on: 2026-08-19 23:50:00
Fetching rows 1-50,000 ...
Fetching rows 50,001-100,000 ...
Fetching rows 100,001-150,000 ...
Fetching rows 150,001-200,000 ...
No more data to fetch.
Fetched 103,680 rows.
API date range: 2026-08-20 00:00:00 to 2026-08-29 23:50:00


In [14]:
data1

,toll_10_minute_block,day_of_week_int,holiday_ind,overnight_ind,region_id,group_id,traffic_volume,dow_sin,dow_cos
0,2025-01-05 00:00:00,1,0,1,1,1,317,7.818315e-01,0.62349
1,2025-01-05 00:10:00,1,0,1,1,1,330,7.818315e-01,0.62349
2,2025-01-05 00:20:00,1,0,1,1,1,300,7.818315e-01,0.62349
3,2025-01-05 00:30:00,1,0,1,1,1,316,7.818315e-01,0.62349
4,2025-01-05 00:40:00,1,0,1,1,1,308,7.818315e-01,0.62349
...,...,...,...,...,...,...,...,...,...
1028155,2026-08-22 23:10:00,7,0,1,7,12,523,-2.449294e-16,1.00000
1028156,2026-08-22 23:20:00,7,0,1,7,12,452,-2.449294e-16,1.00000
1028157,2026-08-22 23:30:00,7,0,1,7,12,488,-2.449294e-16,1.00000
1028158,2026-08-22 23:40:00,7,0,1,7,12,481,-2.449294e-16,1.00000


In [22]:
copy_data = new_data.copy()

In [24]:
copy_data['holiday_ind'] = copy_data['toll_date'].apply(lambda x: 1 if x in holidays.US() else 0)
copy_data['overnight_ind'] = copy_data['time_period'].apply(lambda x: 1 if x == 'Overnight' else 0)

In [32]:
copy_data['day_of_week_int'] = pd.to_numeric(copy_data['day_of_week_int'])

In [33]:
# day of week sin/cos
copy_data['dow_sin'] = np.sin(
    2 * np.pi * copy_data['day_of_week_int'] / 7
)

copy_data['dow_cos'] = np.cos(
    2 * np.pi * copy_data['day_of_week_int'] / 7
)

In [55]:
def process_data(new_data):
    """
    Processes newly pulled API data to match formatting of current processed data.
    """
    # holiday and overnight inds
    new_data['holiday_ind'] = new_data['toll_date'].apply(lambda x: 1 if x in holidays.US() else 0)
    new_data['overnight_ind'] = new_data['time_period'].apply(lambda x: 1 if x == 'Overnight' else 0)

    # group and region ids
    region_mapping = {
        "Brooklyn": 1,
        "Queens": 2,
        "New Jersey": 3,
        "West Side Highway": 4,
        "FDR Drive": 5,
        "West 60th St": 6,
        "East 60th St": 7
    }

    group_mapping = {
        "Brooklyn Bridge": 1,
        "Hugh L. Carey Tunnel": 2,
        "Williamsburg Bridge": 3,
        "Manhattan Bridge": 4,
        "Queensboro Bridge": 5,
        "Queens Midtown Tunnel": 6,
        "Holland Tunnel": 7,
        "Lincoln Tunnel": 8,
        "West Side Highway at 60th St": 9,
        "FDR Drive at 60th St": 10,
        "West 60th St": 11,
        "East 60th St": 12
    }

    new_data['region_id'] = (
        new_data['detection_region']
        .map(region_mapping)
        .fillna(0)
        .astype(int)
    )

    new_data['group_id'] = (
        new_data['detection_group']
        .map(group_mapping)
        .fillna(0)
        .astype(int)
    )

    # day of week sin/cos
    new_data['day_of_week_int'] = pd.to_numeric(new_data['day_of_week_int'])

    new_data['dow_sin'] = np.sin(
        2 * np.pi * new_data['day_of_week_int'] / 7
    )

    new_data['dow_cos'] = np.cos(
        2 * np.pi * new_data['day_of_week_int'] / 7
    )

    # group relevant cols, summing crz_entries + excluded_roadway_entries
    keep_cols = ['toll_10_minute_block','day_of_week_int','holiday_ind','overnight_ind','region_id','group_id','dow_sin','dow_cos']

    new_data['crz_entries'] = pd.to_numeric(new_data['crz_entries'])

    new_data['excluded_roadway_entries'] = pd.to_numeric(new_data['excluded_roadway_entries'])

    grouped_data = (
        new_data
        .groupby(keep_cols,as_index=False)
        .agg(
            crz_entries=('crz_entries','sum'),
            excluded_roadway_entries=('excluded_roadway_entries','sum')
        )
    )

    grouped_data['traffic_volume'] = (
        grouped_data['crz_entries']
        + grouped_data['excluded_roadway_entries']
    )

    grouped_data.drop(['crz_entries','excluded_roadway_entries'],axis=1,inplace=True)

    return grouped_data

In [56]:
data2 = process_data(new_data)

In [52]:
data1 = data1[
    ['toll_10_minute_block','day_of_week_int','holiday_ind','overnight_ind',
    'region_id','group_id','dow_sin','dow_cos','traffic_volume']
]

In [53]:
data1

,toll_10_minute_block,day_of_week_int,holiday_ind,overnight_ind,region_id,group_id,dow_sin,dow_cos,traffic_volume
0,2025-01-05 00:00:00,1,0,1,1,1,7.818315e-01,0.62349,317
1,2025-01-05 00:10:00,1,0,1,1,1,7.818315e-01,0.62349,330
2,2025-01-05 00:20:00,1,0,1,1,1,7.818315e-01,0.62349,300
3,2025-01-05 00:30:00,1,0,1,1,1,7.818315e-01,0.62349,316
4,2025-01-05 00:40:00,1,0,1,1,1,7.818315e-01,0.62349,308
...,...,...,...,...,...,...,...,...,...
1028155,2026-08-22 23:10:00,7,0,1,7,12,-2.449294e-16,1.00000,523
1028156,2026-08-22 23:20:00,7,0,1,7,12,-2.449294e-16,1.00000,452
1028157,2026-08-22 23:30:00,7,0,1,7,12,-2.449294e-16,1.00000,488
1028158,2026-08-22 23:40:00,7,0,1,7,12,-2.449294e-16,1.00000,481


In [46]:
data2

,toll_10_minute_block,day_of_week_int,holiday_ind,overnight_ind,region_id,group_id,dow_sin,dow_cos,traffic_voulme
0,2026-08-20 00:00:00,5,0,1,1,1,-9.749279e-01,-0.222521,242
1,2026-08-20 00:00:00,5,0,1,1,2,-9.749279e-01,-0.222521,19
2,2026-08-20 00:00:00,5,0,1,1,3,-9.749279e-01,-0.222521,191
3,2026-08-20 00:00:00,5,0,1,1,4,-9.749279e-01,-0.222521,114
4,2026-08-20 00:00:00,5,0,1,2,5,-9.749279e-01,-0.222521,99
...,...,...,...,...,...,...,...,...,...
17275,2026-08-29 23:50:00,7,0,1,3,8,-2.449294e-16,1.000000,257
17276,2026-08-29 23:50:00,7,0,1,4,9,-2.449294e-16,1.000000,184
17277,2026-08-29 23:50:00,7,0,1,5,10,-2.449294e-16,1.000000,603
17278,2026-08-29 23:50:00,7,0,1,6,11,-2.449294e-16,1.000000,181


In [ ]:
# All columns from toll_date through detection_region
key_cols = data2.columns[
    data2.columns.get_loc("toll_10_minute_block"):
    data2.columns.get_loc("traffic_volume") + 1
].tolist()

# df1 comes second so its values win on duplicate keys
df = (
    pd.concat([data2, data1], ignore_index=True)
      .drop_duplicates(subset=key_cols, keep="last")
      .reset_index(drop=True)
)

0         2026-08-20 00:00:00
1         2026-08-20 00:00:00
2         2026-08-20 00:00:00
3         2026-08-20 00:10:00
4         2026-08-20 00:20:00
                  ...        
1043776   2026-08-22 23:10:00
1043777   2026-08-22 23:20:00
1043778   2026-08-22 23:30:00
1043779   2026-08-22 23:40:00
1043780   2026-08-22 23:50:00
Name: toll_10_minute_block, Length: 1043781, dtype: datetime64[ns]

In [78]:
df = pd.read_csv('../data/modeling_data.csv')

In [80]:
len(df)

1043781

In [81]:
df['toll_10_minute_block'].min()

'2025-01-05 00:00:00'

In [70]:
df['toll_10_minute_block'].max()

Timestamp('2026-08-29 23:50:00')